# PCA Sensitivity Analysis - Pathology Classification

## Objective
Evaluate the trade-off: does PCA transformation that mitigates site effects hurt pathology classification?

## Experiment Design
- **Methods tested:** raw, sitewise, combat, neurocombat, covbat
- **PCA variants:** none, all (full), 0.99, 0.95, 0.90, 0.80
- **Metric:** Pathology classification AUC (higher = better)

## Key Results

| Method | No PCA | PCA 0.80 |
|--------|--------|----------|
| raw | 0.803 | 0.772 |
| combat | **0.811** | 0.782 |
| sitewise | 0.806 | 0.782 |

## Trade-off Analysis
- **Site MCC reduction:** 0.957 → 0.564 (ComBat with PCA 0.80)
- **Pathology AUC cost:** 0.811 → 0.782 (~3% drop)

The trade-off exists but may be acceptable depending on the use case. PCA helps site-invariance at a modest pathology cost.

## Conclusion
PCA transformation offers a partial solution to the paradox, but a better approach (like DANN) could potentially achieve site-invariance without sacrificing pathology performance.

In [1]:
import os
from pathlib import Path

if Path.cwd().name != 'eeg-site-effects':
os.chdir('../..')
print('Working directory:', Path.cwd())

Working directory: /dmj/fizmed/kchorzela/licencjat/eeg-site-effects


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_PATH_CATBOOST = 'results/tables/05_pca_sensitivity/pca_sensitivity_results_patho_catboost.csv'
RESULTS_PATH_LOGREG   = 'results/tables/05_pca_sensitivity/pca_sensitivity_results_patho_logreg.csv'
FIGURES_DIR  = 'results/figures/05_pca_sensitivity/pathology_classification'

MODELS       = ['catboost', 'logreg']
METHODS      = ['raw', 'sitewise', 'combat', 'neurocombat', 'covbat']
PCA_VARIANTS = ['none', 'all', '0.99', '0.95', '0.9', '0.8']

# --- Shared palette ---
METHOD_COLORS = {
    'raw':         '#1a1a1a',
    'sitewise':    '#e6550d',
    'combat':      'steelblue',
    'neurocombat': '#31a354',
    'covbat':      '#756bb1',
}

PCA_COLORS = {
    'none': '#1a1a1a',
    'all':  '#2166ac',
    '0.99': '#4393c3',
    '0.95': '#92c5de',
    '0.9':  '#f4a582',
    '0.8':  '#d6604d',
}

MODEL_COLORS  = {'catboost': 'steelblue', 'logreg': 'seagreen'}
MODEL_LABELS  = {'catboost': 'CatBoost',  'logreg': 'Logistic Regression'}

# --- Shared figure settings ---
FIGSIZE_WIDE   = (17, 7)
TITLE_FS       = 14
LABEL_FS       = 12
TICK_FS        = 8
LEGEND_FS      = 10
GRID_ALPHA     = 0.5
LINE_ALPHA     = 0.8
SAVE_DPI       = 150

In [ ]:
catboost_df = pd.read_csv(RESULTS_PATH_CATBOOST)
print(catboost_df.shape)
print('methods:', catboost_df['method'].unique())
print('pca_var:', catboost_df['pca_var'].unique())

## CatBoost — AUC per hospital

In [ ]:
os.makedirs(FIGURES_DIR, exist_ok=True)

metric = 'auc'
baseline = catboost_df[(catboost_df['method'] == 'raw') & (catboost_df['pca_var'] == 'none')]
hospitals_sorted = baseline.groupby('hospital')[metric].mean().sort_values(ascending=False).index.values

# --- By method (subplots faceted by PCA variant) ---
fig, axes = plt.subplots(len(PCA_VARIANTS), 1, figsize=(FIGSIZE_WIDE[0], FIGSIZE_WIDE[1] * len(PCA_VARIANTS)), sharey=True)

for ax, pca in zip(axes, PCA_VARIANTS):
    for method in METHODS:
        sub = catboost_df[(catboost_df['method'] == method) & (catboost_df['pca_var'] == pca)]
        vals = sub.set_index('hospital').reindex(hospitals_sorted)[metric]
        ax.plot(range(len(hospitals_sorted)), vals.values, marker='o',
                color=METHOD_COLORS[method], label=method, alpha=LINE_ALPHA)

    ax.set_title(f'CatBoost — PCA = {pca}', fontsize=TITLE_FS)
    ax.set_xticks(range(len(hospitals_sorted)))
    ax.set_xticklabels(hospitals_sorted, rotation=90, fontsize=TICK_FS)
    ax.set_xlabel('Hospital')
    ax.grid(axis='y', linestyle='--', alpha=GRID_ALPHA)
    ax.legend(fontsize=LEGEND_FS)

axes[0].set_ylabel(metric.upper(), fontsize=LABEL_FS)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/patho_auc_catboost_by_method.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## CatBoost — AUC per hospital, grouped by PCA variant

In [ ]:
# --- By PCA (subplots faceted by method) ---
fig, axes = plt.subplots(len(METHODS), 1, figsize=(FIGSIZE_WIDE[0], FIGSIZE_WIDE[1] * len(METHODS)), sharey=True)

for ax, method in zip(axes, METHODS):
    for pca in PCA_VARIANTS:
        sub = catboost_df[(catboost_df['method'] == method) & (catboost_df['pca_var'] == pca)]
        vals = sub.set_index('hospital').reindex(hospitals_sorted)[metric]
        ax.plot(range(len(hospitals_sorted)), vals.values, marker='o',
                color=PCA_COLORS[pca], label=f'pca={pca}', alpha=LINE_ALPHA)

    ax.set_title(f'CatBoost — Method = {method}', fontsize=TITLE_FS)
    ax.set_xticks(range(len(hospitals_sorted)))
    ax.set_xticklabels(hospitals_sorted, rotation=90, fontsize=TICK_FS)
    ax.set_xlabel('Hospital')
    ax.grid(axis='y', linestyle='--', alpha=GRID_ALPHA)
    ax.legend(fontsize=LEGEND_FS)

axes[0].set_ylabel(metric.upper(), fontsize=LABEL_FS)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/patho_auc_catboost_by_pca.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

# Logistic Regression Analysis

In [ ]:
logreg_df = pd.read_csv(RESULTS_PATH_LOGREG)
print(logreg_df.shape)
print('methods:', logreg_df['method'].unique())
print('pca_var:', logreg_df['pca_var'].unique())

In [41]:

metric = 'auc'

baseline = logreg_df[(logreg_df['method'] == 'raw') & (logreg_df['pca_var'] == 'none')]
hospitals_sorted = baseline.groupby('hospital')[metric].mean().sort_values(ascending=False).index.values


## LogReg — Harmonization Effect

In [ ]:
fig, axes = plt.subplots(len(PCA_VARIANTS), 1, figsize=(FIGSIZE_WIDE[0], FIGSIZE_WIDE[1] * len(PCA_VARIANTS)), sharey=True)

for ax, pca in zip(axes, PCA_VARIANTS):
    for method in METHODS:
        sub = logreg_df[(logreg_df['method'] == method) & (logreg_df['pca_var'] == pca)]
        vals = sub.set_index('hospital').reindex(hospitals_sorted)[metric]
        ax.plot(range(len(hospitals_sorted)), vals.values, marker='o',
                color=METHOD_COLORS[method], label=method, alpha=LINE_ALPHA)

    ax.set_title(f'LogReg — PCA = {pca}', fontsize=TITLE_FS)
    ax.set_xticks(range(len(hospitals_sorted)))
    ax.set_xticklabels(hospitals_sorted, rotation=90, fontsize=TICK_FS)
    ax.set_xlabel('Hospital')
    ax.grid(axis='y', linestyle='--', alpha=GRID_ALPHA)
    ax.legend(fontsize=LEGEND_FS)

axes[0].set_ylabel(metric.upper(), fontsize=LABEL_FS)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/patho_auc_logreg_by_method.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## LogReg — PCA Effect, faceted by method

In [ ]:
fig, axes = plt.subplots(len(METHODS), 1, figsize=(FIGSIZE_WIDE[0], FIGSIZE_WIDE[1] * len(METHODS)), sharey=True)

for ax, method in zip(axes, METHODS):
    for pca in PCA_VARIANTS:
        sub = logreg_df[(logreg_df['method'] == method) & (logreg_df['pca_var'] == pca)]
        vals = sub.set_index('hospital').reindex(hospitals_sorted)[metric]
        ax.plot(range(len(hospitals_sorted)), vals.values, marker='o',
                color=PCA_COLORS[pca], label=f'pca={pca}', alpha=LINE_ALPHA)

    ax.set_title(f'LogReg — Method = {method}', fontsize=TITLE_FS)
    ax.set_xticks(range(len(hospitals_sorted)))
    ax.set_xticklabels(hospitals_sorted, rotation=90, fontsize=TICK_FS)
    ax.set_xlabel('Hospital')
    ax.grid(axis='y', linestyle='--', alpha=GRID_ALPHA)
    ax.legend(fontsize=LEGEND_FS)

axes[0].set_ylabel(metric.upper(), fontsize=LABEL_FS)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/patho_auc_logreg_by_pca.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Multi-Model Comparison: Logistic Regression vs CatBoost

In [ ]:
catboost_tagged = catboost_df.assign(model='catboost')
logreg_tagged = logreg_df.assign(model='logreg')
multimodel_df = pd.concat([catboost_tagged, logreg_tagged], ignore_index=True)

In [ ]:
agg = multimodel_df.groupby(['method', 'model', 'pca_var'])['auc'].mean().reset_index()

fig, axes = plt.subplots(1, len(METHODS), figsize=(20, 7), sharey=True)

for ax, method in zip(axes, METHODS):
    for model_name in MODELS:
        subset = agg[(agg['method'] == method) & (agg['model'] == model_name)]
        subset = subset.set_index('pca_var').reindex(PCA_VARIANTS)
        ax.plot(PCA_VARIANTS, subset['auc'].values, marker='o',
                color=MODEL_COLORS[model_name], label=MODEL_LABELS[model_name], alpha=LINE_ALPHA)
    ax.set_title(f'Method = {method}', fontsize=TITLE_FS)
    ax.set_xlabel('PCA variance', fontsize=LABEL_FS)
    ax.tick_params(axis='x', rotation=45, labelsize=TICK_FS)
    ax.grid(axis='y', linestyle='--', alpha=GRID_ALPHA)

axes[0].set_ylabel('Mean AUC', fontsize=LABEL_FS)
axes[-1].legend(title='Model', loc='lower left', fontsize=LEGEND_FS)
fig.suptitle('Pathology classification — CatBoost vs LogReg', fontsize=TITLE_FS + 1, y=1.02)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/patho_auc_multimodel_by_method.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.show()